2025/11
マルチプロセスによる高速化後に作成

In [1]:
import os 
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scipy.stats as stats
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve
import re
import math
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score, recall_score, f1_score
import joblib
import time
from ultralytics import YOLO

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


In [ ]:
# run_process.ipynb のセル

# 1. pip install したライブラリをインポート
import maesyori_fast

# 2. パラメータを定義
BASE_DIR = '/home/data/1104_test' 
MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'
AREA_THRESHOLD = 50000
TARGET_FOLDERS = ['A', 'B', 'C'] 

# 3. 実行
# (if __name__ == "__main__": は引き続き必要)
if __name__ == "__main__":
    start = time.time()
    maesyori_fast.run(
        base_dir=BASE_DIR,
        model_path=MODEL_PATH,
        target_folders=TARGET_FOLDERS,
        area_threshold=AREA_THRESHOLD,
        max_workers=2
    )
current_time = time.time()
print(f"--- [1. 前処理] 完了 (所要時間: {current_time - start:.2f}秒) ---")

In [3]:
import hida_fast
import keijo_fast
import size_module_fast
#判別フェーズ
#特徴量抽出
data = "0107_tomine"
hida_tappleA = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"/home/data/{data}",subfolder="A",method="45rotate",n=9,T=0.4)
result_hidaA = hida_tappleA.run_all()
hida_tappleB = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"/home/data/{data}",subfolder="B",method="45rotate",n=9,T=0.4)
result_hidaB = hida_tappleB.run_all()
hida_tappleC = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"/home/data/{data}",subfolder="C",method="45rotate",n=9,T=0.4)
result_hidaC = hida_tappleC.run_all()

In [ ]:
dfA = pd.DataFrame(result_hidaA, columns=["filename", "R"])
dfB = pd.DataFrame(result_hidaB, columns=["filename", "R"])
dfC = pd.DataFrame(result_hidaC, columns=["filename", "R"])
dfA ["Label"] = "0"
dfB ["Label"] = "1"
dfC ["Label"] = "2"
result_hida = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)
# hida_time = time.time() - start
# print(f"Hida_fastの総処理時間: {hida_time:.2f}秒")

NameError: name 'hida_time' is not defined

In [ ]:
# print(result_hida)

In [ ]:
size_tappleA = size_module_fast.Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="A")
result_sizeA = size_tappleA.run()
size_tappleB = size_module_fast.Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="B")
result_sizeB = size_tappleB.run()
size_tappleC = size_module_fast.Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="C")
result_sizeC = size_tappleC.run()

In [ ]:
dfA = pd.DataFrame(result_sizeA, columns=["filename", "size_count"])                                                    
dfB = pd.DataFrame(result_sizeB, columns=["filename", "size_count"])
dfC = pd.DataFrame(result_sizeC, columns=["filename", "size_count"])
result_size = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)
size_time = time.time() - start
print(f"Size_module_fastの総処理時間: {size_time:.2f}秒")

In [ ]:
keijo_tappleA = keijo_fast.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="A")
result_keijoA = keijo_tappleA.run()
keijo_tappleB = keijo_fast.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="B")
result_keijoB = keijo_tappleB.run()
keijo_tappleC = keijo_fast.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="C")
result_keijoC = keijo_tappleC.run()

In [ ]:
dfA = pd.DataFrame(result_keijoA, columns=["filename", "MSE"])
dfB = pd.DataFrame(result_keijoB, columns=["filename", "MSE"])
dfC = pd.DataFrame(result_keijoC, columns=["filename", "MSE"])
result_keijo = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)
keijo_time = time.time() - start
print(f"Keijo_fastの総処理時間: {keijo_time:.2f}秒")

In [ ]:
df_merged = pd.merge(result_keijo, result_size, on="filename")
df_merged = pd.merge(df_merged, result_hida, on="filename")
df_merged.to_csv(f"/home/data/{data}/feature.csv", index=False)

In [ ]:
# #SVM

# model =" _0827"


# # === 1. データの読み込み ===
# merged_data_csv = f"/home/data/{data}/feature.csv"
# df = pd.read_csv(merged_data_csv)
# # === 2. 特徴量とターゲット変数の定義 ===
# X = df[["MSE", "size_count", "R"]]  # 特徴量
# y = df["Label"]  # 目的変数

# # === 3. 訓練データとテストデータに分割 ===
# # X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=100, random_state=56021, stratify=y)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=56021, stratify=y)
# # 訓練データとテストデータのクラスごとのカウント
# train_counts = y_train.value_counts().sort_index()
# test_counts = y_test.value_counts().sort_index()

# # 結果を表示
# print("Train Data Class Distribution:")
# print(train_counts)
# print("\nTest Data Class Distribution:")
# print(test_counts)


# # === 4. 特徴量の標準化 ===
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# # === 5. SVMの学習 (RBFカーネル) ===
# svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', decision_function_shape='ovr')
# svm_model.fit(X_train, y_train)


# # 学習済みモデルを保存
# model_path = f"svm_model{model}.pkl"
# joblib.dump(svm_model, model_path)

# # 標準化のスケーラーも保存
# scaler_path = f"scaler{model}.pkl"
# joblib.dump(scaler, scaler_path)


# # === 6. 予測と評価 ===
# y_pred = svm_model.predict(X_test)

# # レポートを辞書形式で取得
# report_dict = classification_report(y_test, y_pred, output_dict=True)

# # DataFrameに変換
# report_df = pd.DataFrame(report_dict).transpose()

# # 表示（必要に応じて四捨五入）
# print(report_df.round(4))

# # y_testが持つインデックスを使い、元のdfからテストデータに該当する行だけを抽出
# test_data_with_results = df.loc[y_test.index].copy()

# # 抽出したテストデータに、予測結果の列を追加
# # これで左辺(test_data_with_results)と右辺(y_pred)の行数が一致する
# test_data_with_results['Predicted_Label'] = y_pred

# # 結果を新しいCSVファイルとして保存
# output_path = f"/home/data/{data}/BBBpredicted_results{model}.csv"
# test_data_with_results.to_csv(output_path, index=False)


# # === 7. 混同行列の可視化 ===
# cm = confusion_matrix(y_test, y_pred)
# plt.figure(figsize=(6, 5))
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0, 1, 2], yticklabels=[0, 1, 2])
# plt.xlabel("Predicted Label")
# plt.ylabel("True Label")
# plt.title("Confusion Matrix")
# plt.show()


In [ ]:
end = time.time()
print(f"総処理時間: {end - start:.2f}秒")

In [ ]:

# === モデルとスケーラーの読み込み ===
model = "_jikuari_tai"
model_path = "../svm_model.pkl"
scaler_path = "../scaler.pkl"
df_merged.to_csv(f'/home/data/{data}/feature{model}.csv', index=False)
svm_model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

# === 新しいデータの読み込み ===
# df = pd.DataFrame(merged)
df = pd.read_csv(f'/home/data/{data}/feature{model}.csv')
df = pd.DataFrame(df)

# === 特徴量の抽出と標準化 ===
X_new = df[["MSE", "size_count", "R"]]  # 学習時と同じ特徴量を使用
X_new = scaler.transform(X_new)  # 標準化

# === 予測 ===
y_pred_new = svm_model.predict(X_new)

# 結果をDataFrameに追加
df["Predicted_Label"] = y_pred_new

# 予測結果の確認
print([["MSE", "size_count", "R", "Predicted_Label"]])

# CSVとして保存（オプション）
df.to_csv(f"/home/data/{data}predicted{model}.csv", index=False)


pyのほうがバグってたのでこっちで

In [ ]:
import os 
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scipy.stats as stats
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve
import re
import math
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score, recall_score, f1_score
import joblib
import time
from ultralytics import YOLO
import hida_fast
import keijo_fast
import size_module_fast
import maesyori_fast

In [ ]:
# 1. pip install したライブラリをインポート

# (import文は元のまま)

# === 全体時間の計測開始 ===
print("========== 全処理開始 ==========")
overall_start_time = time.time()
last_timestamp = overall_start_time # 各ステップの時間を測るための中間地点

# 2. パラメータを定義
BASE_DIR = '/home/data/1104_test' 
MODEL_PATH = '/home/YOLO/0708_maesyori/datasets/train/weights/best.pt'
AREA_THRESHOLD = 50000
TARGET_FOLDERS = ['A', 'B', 'C'] 

In [ ]:
# 3. 実行
# (if __name__ == "__main__": は引き続き必要)
if __name__ == "__main__":
    
    # -----------------------------------------------------------------
    # [1. 前処理 (maesyori_fast)]
    # -----------------------------------------------------------------
    print("\n--- [1. 前処理 (YOLOセグメンテーション等)] 開始 ---")
    maesyori_fast.run(
        base_dir=BASE_DIR,
        model_path=MODEL_PATH,
        target_folders=TARGET_FOLDERS,
        area_threshold=AREA_THRESHOLD,
        max_workers=2
    )
    
    current_time = time.time()
    print(f"--- [1. 前処理] 完了 (所要時間: {current_time - last_timestamp:.2f}秒) ---")
    last_timestamp = current_time
    # -----------------------------------------------------------------

In [ ]:
    #判別フェーズ
    #特徴量抽出
    data = "1104_test"
    
    # -----------------------------------------------------------------
    # [2. 特徴量抽出 (hida_fast)]
    # -----------------------------------------------------------------
    print("\n--- [2. 特徴量抽出 (hida)] 開始 ---")
    hida_tappleA = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"/home/data/{data}",subfolder="A",method="45rotate",n=9,T=0.4)
    result_hidaA = hida_tappleA.run_all()
    hida_tappleB = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"/home/data/{data}",subfolder="B",method="45rotate",n=9,T=0.4)
    result_hidaB = hida_tappleB.run_all()
    hida_tappleC = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"/home/data/{data}",subfolder="C",method="45rotate",n=9,T=0.4)
    result_hidaC = hida_tappleC.run_all()

    dfA = pd.DataFrame(result_hidaA, columns=["filename", "R"])
    dfB = pd.DataFrame(result_hidaB, columns=["filename", "R"])
    dfC = pd.DataFrame(result_hidaC, columns=["filename", "R"])
    dfA ["Label"] = "0"
    dfB ["Label"] = "1"
    dfC ["Label"] = "2"
    result_hida = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)
    
    current_time = time.time()
    print(f"--- [2. 特徴量抽出 (hida)] 完了 (所要時間: {current_time - last_timestamp:.2f}秒) ---")
    last_timestamp = current_time
    # -----------------------------------------------------------------

In [ ]:
    # -----------------------------------------------------------------
    # [3. 特徴量抽出 (size_module_fast)]
    # -----------------------------------------------------------------
    print("\n--- [3. 特徴量抽出 (size)] 開始 ---")
    size_tappleA = size_module_fast.Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="A")
    result_sizeA = size_tappleA.run()
    size_tappleB = size_module_fast.Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="B")
    result_sizeB = size_tappleB.run()
    size_tappleC = size_module_fast.Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="C")
    result_sizeC = size_tappleC.run()

    dfA = pd.DataFrame(result_sizeA, columns=["filename", "size_count"]) 
    dfB = pd.DataFrame(result_sizeB, columns=["filename", "size_count"])
    dfC = pd.DataFrame(result_sizeC, columns=["filename", "size_count"])
    result_size = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)

    current_time = time.time()
    print(f"--- [3. 特徴量抽出 (size)] 完了 (所要時間: {current_time - last_timestamp:.2f}秒) ---")
    last_timestamp = current_time
    # -----------------------------------------------------------------

In [ ]:
    # -----------------------------------------------------------------
    # [4. 特徴量抽出 (keijo_fast)]
    # -----------------------------------------------------------------
    print("\n--- [4. 特徴量抽出 (keijo)] 開始 ---")
    keijo_tappleA = keijo_fast.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="A")
    result_keijoA = keijo_tappleA.run()
    keijo_tappleB = keijo_fast.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="B")
    result_keijoB = keijo_tappleB.run()
    keijo_tappleC = keijo_fast.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="C")
    result_keijoC = keijo_tappleC.run()

    dfA = pd.DataFrame(result_keijoA, columns=["filename", "MSE"])
    dfB = pd.DataFrame(result_keijoB, columns=["filename", "MSE"])
    dfC = pd.DataFrame(result_keijoC, columns=["filename", "MSE"])
    result_keijo = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)

    current_time = time.time()
    print(f"--- [4. 特徴量抽出 (keijo)] 完了 (所要時間: {current_time - last_timestamp:.2f}秒) ---")
    last_timestamp = current_time
    # -----------------------------------------------------------------

In [ ]:
    # -----------------------------------------------------------------
    # [5. 特徴量の結合]
    # -----------------------------------------------------------------
    print("\n--- [5. 特徴量の結合] 開始 ---")
    df_merged = pd.merge(result_keijo, result_size, on="filename")
    df_merged = pd.merge(df_merged, result_hida, on="filename")
    # df_merged.to_csv(f"/home/data/{data}/feature.csv", index=False) # コメントアウトされていたためそのまま

    current_time = time.time()
    print(f"--- [5. 特徴量の結合] 完了 (所要時間: {current_time - last_timestamp:.2f}秒) ---")
    last_timestamp = current_time
    # -----------------------------------------------------------------

In [ ]:
    # -----------------------------------------------------------------
    # [6. SVMによる予測]
    # -----------------------------------------------------------------
    print("\n--- [6. SVMによる予測] 開始 ---")
    # === モデルとスケーラーの読み込み ===
    model = "_jikuari_tai"
    model_path = "../svm_model.pkl"
    scaler_path = "../scaler.pkl"
    # df_merged.to_csv(f'/home/data/{data}/feature{model}.csv', index=False) # コメントアウトされていたためそのまま
    svm_model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)

    # === 新しいデータの読み込み ===
    # df = pd.DataFrame(merged)
    # df = pd.read_csv(f'/home/data/{data}/feature{model}.csv') # コメントアウトされていたためそのまま
    df = pd.DataFrame(df_merged)

    # === 特徴量の抽出と標準化 ===
    X_new = df[["MSE", "size_count", "R"]] 
    X_new = scaler.transform(X_new) 

    # === 予測 ===
    y_pred_new = svm_model.predict(X_new)

    # 結果をDataFrameに追加
    df["Predicted_Label"] = y_pred_new

    # CSVとして保存（オプション）
    df.to_csv(f"/home/data/{data}predicted{model}.csv", index=False)

    current_time = time.time()
    print(f"--- [6. SVMによる予測] 完了 (所要時間: {current_time - last_timestamp:.2f}秒) ---")
    # -----------------------------------------------------------------

In [ ]:
# === 全体時間の計測終了 ===
overall_end_time = time.time()
print(f"\n========== 全処理完了 ==========")
print(f"(総所要時間: {overall_end_time - overall_start_time:.2f}秒)")

# (※元の start, end 変数での計測は、上記の計測に統合されました)